In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Verify GPU

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
GPU Available: True
GPU: Tesla T4


In [5]:
!pip install transformers datasets accelerate wandb huggingface_hub

## Verify Packages

In [11]:
import transformers
import datasets
import torch

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)

Transformers: 5.0.0
Datasets: 4.8.5
Torch: 2.10.0+cu128


## Load Secrets and Check Login

In [14]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os
import wandb

secrets = UserSecretsClient()

os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

wandb.login()

HF_TOKEN = secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN)

print("Login successful")


Login successful


## Load Dataset

In [15]:
# 1. Load Dataset
from datasets import load_dataset
import json

dataset = load_dataset("stanfordnlp/imdb", token=HF_TOKEN)

# 2. Dataset Inspection
print(dataset)

print("Train size:", len(dataset["train"]))
print("Test size:", len(dataset["test"]))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Train size: 25000
Test size: 25000


## Load Tokenizer

In [16]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded


## Tokenize Dataset

In [17]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed")


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenization completed


## Remove Text Column

In [18]:
tokenized_dataset = tokenized_dataset.remove_columns(
    ["text"]
)

tokenized_dataset.set_format("torch")

print("Dataset formatted")

Dataset formatted


# Verify Tokenized Dataset

In [19]:
print(tokenized_dataset["train"][0])


{'label': tensor(0), 'input_ids': tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
         2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
         2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
         2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
         1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
         2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
         6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
         1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
         5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
        14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
         1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
         2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779,
        25430, 14728,  2245,  

In [20]:
print(tokenized_dataset["train"][0].keys())

dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])


# Create Small Dataset Sample(For Testing)

In [21]:
# small_train = (
#     tokenized_dataset["train"]
#     .shuffle(seed=42)
#     .select(range(2000))
# )

# small_test = (
#     tokenized_dataset["test"]
#     .shuffle(seed=42)
#     .select(range(500))
# )

# print(len(small_train))
# print(len(small_test))


## Create Dataset

In [22]:
train_data = tokenized_dataset["train"]
test_data = tokenized_dataset["test"]

print(len(train_data))
print(len(test_data))

25000
25000


# Load Model

In [23]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("Model loaded")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


# Metrics:

In [24]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy":
            accuracy_score(labels, preds),

        "f1":
            f1_score(
                labels,
                preds,
                average="weighted"
            )
    }

# Initialize W&B

In [25]:
import wandb

wandb.init(
    project="mlops-imdb-sentiment",

    name="run-v1",

    config={
        "epochs":3,
        "batch_size":16,
        "learning_rate":3e-5,
        "version":"v1"
    }
)

# Training arguments:

In [26]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="wandb",

    run_name="run-v1"
)

## Trainer

In [27]:
from transformers import Trainer

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_data,

    eval_dataset=test_data,

    compute_metrics=compute_metrics
)


print(tokenized_dataset["train"][0].keys)

print("Trainer created")


<built-in method keys of dict object at 0x7ef6573e3840>
Trainer created


# Train

In [28]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.610029,0.525346,0.889440,0.888874
2,0.278059,0.627100,0.905840,0.905734
3,0.144714,0.779851,0.913280,0.913273


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2346, training_loss=0.3271724704179821, metrics={'train_runtime': 1324.3763, 'train_samples_per_second': 56.63, 'train_steps_per_second': 1.771, 'total_flos': 4967527449600000.0, 'train_loss': 0.3271724704179821, 'epoch': 3.0})

## Evaluate

In [29]:
results = trainer.evaluate()

print(results)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.5254761576652527, 'eval_accuracy': 0.88944, 'eval_f1': 0.8888728432639231, 'eval_runtime': 110.807, 'eval_samples_per_second': 225.618, 'eval_steps_per_second': 7.057, 'epoch': 3.0}


## Finish W&B

In [30]:
wandb.finish()

eval/accuracy,▁▆█▁
eval/f1,▁▆█▁
eval/loss,▁▄█▁
eval/runtime,▂▁█▆
eval/samples_per_second,▇█▁▃
eval/steps_per_second,▇█▁▃
train/epoch,▁▂▃▅▅▇███
train/global_step,▁▂▃▅▅▇███
train/grad_norm,▁▁▃█
train/learning_rate,█▆▃▁
+1,...


## Push Best Model to Hugging Face

In [31]:
# Adding the right labelling
model.config.id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

model.config.label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

In [ ]:
model.push_to_hub(
    "rakeshlrng/imdb-sentiment"
)

tokenizer.push_to_hub(
    "rakeshlrng/imdb-sentiment"
)